[Download support files](/downloads/Week_01.zip) · [Run in Colab](https://colab.research.google.com/)

Upload the notebook and matching ZIP in Colab as described in the setup guide. The website displays code without executing it.


[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

[Course home](../README.md) · [Setup](../SETUP.md) · [Assignment guides](../assignments/README.md) · [Course links](../COURSE_LINKS.md)

The badge opens Colab. Choose **File → Open notebook → Upload** and upload this `.ipynb` file from your downloaded package. In Colab, view the downloaded guides separately. Run the setup cell below; in Colab, upload the matching weekly ZIP when prompted to load its support files. Locally, open this notebook from the extracted `learning_materials/` directory. Keep your own saved copy of assessed work.



(week-01-start)=
# Week 01 · Values, types and execution

[Start here](#week-01-start) · [Session 1](#week-01-session-1) · [Session 2](#week-01-session-2) · [Project studio](#week-01-project-studio) · [Finish](#week-01-finish)

**This notebook is your working home for the whole week.** Read the explanations, run the examples, write the practice here, then continue to the project. [Python Basics](Python_Basics.ipynb) is an optional syntax lookup; you do not need to leave this notebook to complete the core work.

**Required route:** read → predict → run → modify → check → next. Complete the 3 core practice tasks and the marked project work. Optional follow-ups are clearly labelled and can wait.

**Save:** keep your edited `Week_01.ipynb`, any files you create, and a separate backup. Use the instructor's announced private route for completed assessed work. The exact deadline/file-naming rule will be announced; shared Issues are for small questions, not complete submissions.

**Run order:** start with a fresh kernel and run downward. Teaching examples are complete; cells tagged or labelled as your work intentionally remain unfinished. Running the untouched starter is not completion.


### Prepare your workspace

This setup helper is not a programming exercise. Run it once per session; in Colab, upload the complete weekly ZIP when prompted. Then continue with the lesson. Later student-work cells are intentionally unfinished, so read each task before running it.


In [ ]:
# @title Prepare the course files (run once per session)
"""Prepare the supplied course files in Jupyter or a temporary Colab runtime.

The release builder embeds this source in a notebook setup cell, then calls
prepare_course_workspace(). Importing this module alone does not upload files.
"""
from pathlib import Path, PurePosixPath
import io
import os
import shutil
import stat
import tempfile
import zipfile


_REQUIRED = (
    "programs/pickup_app.py", "programs/first_pickup.py",
    "programs/rescue_cli.py", "programs/rescue_models.py",
    "programs/test_pickup_app.py", "programs/test_rescue_models.py",
    "final-data/offer_batches.csv", "final-data/pickup_requests.csv",
    "final-data/rescue_events.csv", "assets/diagrams/python_execution_modes.svg",
)
_ALLOWED = {".py", ".csv", ".md", ".svg", ".png", ".jpg", ".jpeg", ".ipynb", ".json"}


def is_course_directory(folder):
    """Recognize the student directory without requiring a particular week."""
    folder = Path(folder)
    return all((folder / name).is_file() for name in _REQUIRED)


def extract_course_zip(data, destination_parent=None):
    """Extract a unique learning_materials/ tree into a NEW directory and return its path.

    Supports the downloadable weekly ZIP and GitHub's repository ZIP. Other
    directories are ignored. Existing work is never replaced. No archive code
    runs during extraction; running notebook cells remains a separate action.
    """
    with zipfile.ZipFile(io.BytesIO(data)) as archive:
        candidates = set()
        for member in archive.infolist():
            parts = PurePosixPath(member.filename).parts
            if len(parts) >= 3 and parts[-3:] == ("learning_materials", "programs", "pickup_app.py"):
                candidates.add(parts[:-2])
        if len(candidates) != 1:
            raise ValueError("Choose a course or weekly ZIP with exactly one learning_materials/ folder. Do not upload the instructor workspace ZIP.")
        prefix = next(iter(candidates))
        files, paths, total = [], set(), 0
        for member in archive.infolist():
            raw = member.filename
            parts = PurePosixPath(raw).parts
            if parts[:len(prefix)] != prefix:
                continue
            if "\\" in raw or raw.startswith("/") or any(p in ("..", ".") or ":" in p for p in parts):
                raise ValueError("The course ZIP contains an unsafe file path.")
            relative = PurePosixPath(*parts[len(prefix):])
            if member.is_dir() or not relative.parts:
                continue
            if stat.S_ISLNK(member.external_attr >> 16):
                raise ValueError("The course ZIP contains a symbolic link.")
            if any(p.startswith(".") or p == "__pycache__" for p in relative.parts):
                continue
            if relative.suffix.lower() not in _ALLOWED:
                continue
            if str(relative) in paths:
                raise ValueError("The course ZIP contains duplicate file paths.")
            paths.add(str(relative))
            total += member.file_size
            if total > 100 * 1024 * 1024 or len(files) >= 3000:
                raise ValueError("The course student files exceed the expected package size.")
            files.append((member, relative))
        if not all(name in paths for name in _REQUIRED):
            raise ValueError("The ZIP is missing required programs, data or diagram files. Download the complete course package.")
        session = Path(tempfile.mkdtemp(prefix="campus_food_rescue_", dir=destination_parent))
        student = session / "learning_materials"
        try:
            for member, relative in files:
                target = student.joinpath(*relative.parts)
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(member) as source, target.open("xb") as output:
                    shutil.copyfileobj(source, output)
        except Exception:
            shutil.rmtree(session)
            raise
    return student.resolve()


def prepare_course_workspace(force_upload=False):
    """Use a local student directory, or upload a course ZIP in Google Colab.

    force_upload=True creates a fresh Colab copy when course files are updated.
    It leaves the previous copy intact. Save your work before switching copies.
    """
    try:
        from google.colab import files
    except ImportError:
        for candidate in (Path.cwd(), Path.cwd() / "learning_materials"):
            if is_course_directory(candidate):
                os.chdir(candidate)
                print("Course files ready:", Path.cwd())
                return Path.cwd()
        raise FileNotFoundError("Open this notebook from the course learning_materials/ folder. Keep programs/, assets/ and final-data/ beside it; see SETUP.md.")
    if not force_upload and is_course_directory(Path.cwd()):
        print("Reusing this session's course files:", Path.cwd())
        return Path.cwd()
    print("Upload ONE course ZIP downloaded from GitHub or this week's course page.")
    print("Only its learning_materials/ files will be copied into a fresh temporary folder.")
    previous = Path.cwd()
    # Colab writes uploaded bytes into its current folder before returning them.
    # Keep that operation away from an existing notebook/program workspace.
    with tempfile.TemporaryDirectory(prefix="course_upload_") as upload_folder:
        try:
            os.chdir(upload_folder)
            uploaded = files.upload()
        finally:
            os.chdir(previous)
    if len(uploaded) != 1:
        raise ValueError("Select exactly one course ZIP, then run this setup cell again.")
    name, data = next(iter(uploaded.items()))
    if not name.lower().endswith(".zip"):
        raise ValueError("Choose the .zip package, not an individual notebook or Python file.")
    folder = extract_course_zip(data, destination_parent="/content" if Path("/content").is_dir() else None)
    os.chdir(folder)
    print("Course files ready:", folder)
    print("Runtime files are temporary. Save your notebook and download any created files before leaving.")
    return folder

prepare_course_workspace()


## This week: required, supplied and optional

Follow Start → Sessions 1–3 → Finish in this notebook. All required explanations and exercises are here; Python Basics is optional lookup.

- **Required evidence:** Complete the local values, arithmetic and execution-order practice; then check the one-batch plan for 1, 3 and 0 containers and explain one label and coordinator question.
- **Already supplied:** The examples, diagram, starter values and saved-file program are supplied.
- **Optional extension:** Extra examples or syntax lookup in Python Basics.
- **Typical work outside class:** 30–45 minutes. This estimate includes assigned research preparation; ask early if access or an unclear task takes much longer.
- **Research this week:** Prepare Research 1 for Week 2.

**Stop point:** save the required work and a backup, and check that you can explain your results. Optional work can wait.



(week-01-session-1)=
## Session 1 · Python foundations

**Read:** values, types and execution.

A **variable name** refers to a value. `count = 4` assigns an integer; `"count"` is literal text. Python evaluates the right side of an assignment first, then associates the result with the name on the left.

An **expression** combines values, for example `3 * 4`. Quoted `"3"` is a string, so `"3" * 4` repeats text. `print()` displays a result. A comment beginning with `#` explains code without executing.

Python executes statements in order. Changing an input does not automatically recalculate an earlier expression. Editing a notebook cell without running it also leaves the old output visible. A `.py` file runs in a fresh interpreter, without the notebook's remembered variables.

| Syntax | Meaning | Use |
|---|---|---|
| `name = value` | Assign a value | Keep an input for later |
| `3 * 4` | Multiply numbers | Calculate a quantity |
| `print("Total:", total)` | Display a value | Label output and units |
| `# explanation` | Comment; not executed | Explain an assumption |


**Predict → run:** use the examples below. **Check:** explain a returned value, output or error in your own words. **Next:** [Session 2](#week-01-session-2).

Optional quick lookup: [§00](Python_Basics.ipynb#python-basics-b00) · [§01](Python_Basics.ipynb#python-basics-b01) · [§02](Python_Basics.ipynb#python-basics-b02) · [§03](Python_Basics.ipynb#python-basics-b03) · [§14](Python_Basics.ipynb#python-basics-b14).


### Read → predict → run: Assign, calculate and print

Write a prediction before running. Then trace which statement produced the result. The code is supplied; make your own change during Session 2.


In [ ]:
description = "Practice meal containers"
portions_per_container = 3
containers = 4
planned = portions_per_container * containers
print(description)
print("Containers:", containers)
print("Planned portions:", planned)
assert planned == 12


### Read → predict → run: Execution order: when is a value recalculated?

Write a prediction before running. Then trace which statement produced the result. The code is supplied; make your own change during Session 2.


In [ ]:
portions_per_container = 3
containers = 4
planned = portions_per_container * containers
containers = 2
print("Before:", planned)
planned = portions_per_container * containers
print("After:", planned)
assert planned == 6


**Check → next:** explain one example without reading its output, then open [Session 2](#week-01-session-2).



(week-01-session-2)=
## Session 2 · Hands-on practice

(week-01-practice-week-01)=
**Required:** tasks 1–3 below. The follow-up section is optional.

For each core task: **predict** before execution → **run** the supplied example → **modify** one input or statement → **check** the result → **explain** the first difference. Keep your answer in this notebook. Use the Session 1 examples above; copy only the code you need.


### Practice tasks

1. **Change one value:** Use 3 portions/container. Try 4, then 2 containers. Predict before running.

2. **Trace execution:** Change only the description. Then change the count. Which lines need rerunning?

3. **Run a saved file:** Copy first_pickup.py. Set the count to 3. Save, run, compare.


### Your core practice record

For each required task, write the input/change, your expected result, the observed result and one explanation. Save your edited practice here.


In [ ]:
# Task 1: prediction, code change and observation.
# Task 2: prediction, code change and observation.
# Task 3: prediction, code change and observation.


### Required saved-file check

Copy [programs/first_pickup.py](programs/first_pickup.py) to a separate practice file. Change the count to 3, save it, then run that copy from a terminal. Use the course interpreter and the actual filename you saved. From `learning_materials/`, the supplied original runs as:

```text
python programs/first_pickup.py
```

A terminal command is not a Python statement. Your saved copy runs in a fresh process, so it cannot use variables remembered only in this notebook. Keep the command, prediction and observed output in your practice record.


### Optional follow-up · only after the core checks

The following existing exploration is optional. Use it when it answers a question or the instructor selects it. It is not extra required homework.


**Predict names, literals and repeated text**

Before running, predict the stored count, the literal word and the type name. Then compare 3 * 4 with "3" * 4. Explain why syntactically valid Python may perform a different operation from the one you intended.


**Context:** Standalone Python; no project helper or fixture is required.


In [ ]:
containers = 4
print(containers)
print("containers")
print(type(containers).__name__)

print(3 * 4)
print("3" * 4)
assert 3 * 4 == 12


In [ ]:
# Work on the selected follow-up here.


### Check before moving on

- **Old output:** Rerun assignments and calculation.
- **Repeated text:** Inspect quotes and input types.
- **Unclear meaning:** Name the unit and planning status.

Use the explanations and examples above to check your attempt. **Next:** [Project studio](#week-01-project-studio).



(week-01-project-studio)=
## Session 3 · Project studio

**Read:** the project context below. **Predict:** write an expected result before implementing. **Run/modify:** complete the marked task using the supplied starting data. **Check:** compare evidence with a peer and retain a revision. **Next:** [Finish and save](#week-01-finish).

A suggested rhythm is 10 minutes for the design briefing, 55 for implementation/thinking, 15 for peer checks and 10 for evidence/reflection. Each week supplies its own data; an unfinished earlier notebook is not a prerequisite.


### Project context and working rules

How many portions are we planning, and what does that number establish?

Campus Food Rescue is a fictional Cambodian university volunteer team coordinating with nearby food businesses and community partner coordinators. All names, messages, quantities and approvals are invented. These lessons teach coordination software, not food-safety certification. A human coordinator approves suitability, arrangements and a collection window; code records the supplied decision.

Open in Python 3/Jupyter with a fresh kernel and run from the top. Each week supplies its own data. A planned portion is a declared estimate, not evidence of nutrition, collection, delivery or a distinct person served.

### A donor estimate becomes a planning note

A nearby food business tells the university's donor liaison that each labelled container has three planned portions. Four containers are proposed for a pickup. The volunteer desk needs a readable note; the partner coordinator needs the unit and the meaning of the quantity. The first program will multiply declared portions per container by container count. It will not decide suitability or record a delivery.

This week you learn to name inputs, use integer arithmetic, inspect output and run a saved Python file. The coordinator remains responsible for review and arrangements. We will keep unanswered questions visible instead of replacing them with guessed numbers.

![Declared container contents and count become a labelled planning estimate; a person checks the result.](assets/diagrams/w01_planning.svg)

The learning route is read → predict → run → inspect → change → explain. Worked examples use a practice plan; the independent task uses different inputs.

In [ ]:
# @title Show this lesson’s diagram(s) in Colab
import sys
if "google.colab" in sys.modules:
    from IPython.display import SVG, display
    display(SVG(filename='assets/diagrams/w01_planning.svg'))


### Project discussion and working routine

Follow the message from donor liaison to volunteer desk and partner coordinator. Open, save and reopen the notebook. Read an answered question in the course repository’s GitHub Issues when the course repository is available, using the [class question guide](CLASS_QA.md). If access or connectivity is unavailable, draft a local question with a small example and ask during class or through the separately announced private contact route.



### Think before coding

Name one input ambiguity and one misleading output the team should avoid. State an unresolved question for a coordinator.

**Your response:** write your prediction, observation and explanation here.

### A labelled planning estimate

| Value | Unit | Meaning |
|---|---|---|
| portions_per_container = 3 | Estimated portions/container | Declared contents of one container |
| containers = 4 | Containers | Number proposed for this plan |
| planned_portions = 12 | Estimated portions | Product of the two inputs |

The units cancel appropriately: portions/container × containers = portions. None of these values counts distinct people.

In [ ]:
description = "Practice meal containers"
portions_per_container = 3
containers = 4
planned_portions = portions_per_container * containers
print(description)
print("Containers:", containers)
print("Planned portions:", planned_portions)
assert planned_portions == 12

### Independent lab: one batch planning note

Use a declared estimate of 2 portions per container for practice bread packs. Complete the starter to calculate and label plans for 1, 3 and 0 containers. Ask whether zero is acceptable for a planning draft. A real reservation will later require positive counts. Keep the declaration, input count and result visible.

In [ ]:
description = "Practice bread packs"
portions_per_container = 2
containers = 1
print(description, containers)
# Calculate planned_portions and label the units.
# Check container counts 1, 3 and 0; document the planning assumption.

### Peer review, reflection and evidence

Record one input/prediction/observation, one revision after a peer trial, and one unanswered coordinator question. Explain variable versus literal text and why an edited cell can display an old result. If alone, use a hypothetical reader and label that assumption.

**Your response:** write your prediction, observation and explanation here.

**Individual evidence:** a runnable one-batch planning note, predictions for all three counts, one clarified label and one unresolved coordinator question.

**Next week:** the arithmetic is usable, but the volunteer desk asks whether the plan is ready to schedule and whether it needs a team. We need a human review flag and an explicit branching rule.


(week-01-research)=
### Research assignment 1 for Week 2 · 5%

Investigate three connected questions: What do you want to learn in this course? What is one real information system that supports a decision or service? What should an engineer contribute when generative AI can suggest text or code? Submit your sources, one concrete system example and your own reasoning. Use slides, a short report or another clear format. Presentation is not required for normal full marks.

**Preparation time:** 15–20 minutes, included in this week's outside-class estimate.

AI is optional. If you use it, ask follow-up questions or challenge a claim, then include an accessible conversation link, screenshots or relevant excerpts together with your sources. Do not expose personal data or submit an unverified AI claim as fact.

**Your response:** write your prediction, observation and explanation here.


(week-01-finish)=
## Finish and save

- [ ] I saved my predictions and changes for the three Python practice tasks.
- [ ] My project note calculates 1, 3 and 0 containers with the stated units.
- [ ] I recorded a clarified label and an unanswered coordinator question.
- [ ] I know Research 1 is prepared for Week 2.

**Save now:** save this notebook and your created files, then keep a backup. Completed work goes through the announced private submission route; do not post full assessed solutions in shared Issues. If something is unfinished, record the exact input, expectation, observation and next step.

[Research preparation](#week-01-research) · [Start here](#week-01-start) · [Session 1](#week-01-session-1) · [Session 2](#week-01-session-2) · [Project studio](#week-01-project-studio) · [Finish](#week-01-finish)
